In [ ]:
#!/usr/bin/env python3
"""Large-sample anamorphic-nonce detectability experiment for Google Colab.

This script deliberately measures only statistical detectability.  CPU timing
and protocol-performance measurements must remain on the laptop platform used
in the paper.  For every embedding rate, it generates independent normal and
anamorphic datasets, preserves session boundaries during classifier splitting,
and evaluates both linear and nonlinear distinguishers.
"""

from __future__ import annotations

import hashlib
import hmac
import json
import math
import platform
import secrets
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import sklearn
from scipy.stats import chisquare, entropy as scipy_entropy, ks_2samp
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score


# Paper experiment configuration.
N_PER_MODE = 100_000
ELL_VALUES = (1, 2, 4, 8)
NONCE_LEN = 12
RECORDS_PER_SESSION = 10
MAX_TRIALS = 65_536
BASE_SEED = 42
SPLIT_SEEDS = tuple(range(42, 52))
OUTPUT_DIR = Path(
    "/content/drive/MyDrive/Anamorphic100k/experimental_results_100k"
)
FIGURE_DIR = Path(
    "/content/drive/MyDrive/Anamorphic100k/experimental_figures_100k"
)

def prf_prefix(
    key: bytes,
    transcript_hash: bytes,
    record_number: int,
    nonce: bytes,
    ell: int,
) -> int:
    message = b"ano" + transcript_hash + record_number.to_bytes(8, "big") + nonce
    digest = hmac.new(key, message, hashlib.sha256).digest()
    return int.from_bytes(digest, "big") >> (len(digest) * 8 - ell)


def anamorphic_encode(
    key: bytes,
    transcript_hash: bytes,
    record_number: int,
    hidden_value: int,
    ell: int,
) -> tuple[bytes, int]:
    for trial in range(1, MAX_TRIALS + 1):
        nonce = secrets.token_bytes(NONCE_LEN)
        if prf_prefix(key, transcript_hash, record_number, nonce, ell) == hidden_value:
            return nonce, trial
    raise RuntimeError(
        f"AnoEnc exceeded MAX_TRIALS={MAX_TRIALS} for ell={ell}."
    )


def generate_dataset(
    mode: str,
    ell: int,
    rng: np.random.Generator,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Return nonce bytes, session IDs, and rejection-sampling trial counts."""
    if mode not in {"normal", "anamorphic"}:
        raise ValueError("mode must be 'normal' or 'anamorphic'")

    nonces = np.empty((N_PER_MODE, NONCE_LEN), dtype=np.uint8)
    groups = np.empty(N_PER_MODE, dtype=np.int32)
    trials = np.ones(N_PER_MODE, dtype=np.int32)
    index = 0
    session_id = 0

    while index < N_PER_MODE:
        key = secrets.token_bytes(32)
        transcript_hash = secrets.token_bytes(32)
        for local_record in range(RECORDS_PER_SESSION):
            if index >= N_PER_MODE:
                break
            record_number = session_id * RECORDS_PER_SESSION + local_record
            if mode == "normal":
                nonce = secrets.token_bytes(NONCE_LEN)
                used_trials = 1
            else:
                hidden_value = int(rng.integers(0, 1 << ell))
                nonce, used_trials = anamorphic_encode(
                    key, transcript_hash, record_number, hidden_value, ell
                )
            nonces[index] = np.frombuffer(nonce, dtype=np.uint8)
            groups[index] = session_id
            trials[index] = used_trials
            index += 1
        session_id += 1

    return nonces, groups, trials


def byte_counts(nonces: np.ndarray) -> np.ndarray:
    return np.bincount(nonces.reshape(-1), minlength=256).astype(np.float64)


def distribution_metrics(nonces: np.ndarray) -> dict[str, float]:
    counts = byte_counts(nonces)
    probabilities = counts / counts.sum()
    nonzero = probabilities[probabilities > 0]
    bits = np.unpackbits(nonces, axis=1)
    position_frequencies = bits.mean(axis=0)
    chi = chisquare(counts)
    return {
        "shannon_entropy_bits_per_byte": float(scipy_entropy(nonzero, base=2)),
        "min_entropy_bits_per_byte": float(-math.log2(probabilities.max())),
        "bit_one_frequency": float(bits.mean()),
        "max_bit_position_bias": float(np.max(np.abs(position_frequencies - 0.5))),
        "chi2_statistic": float(chi.statistic),
        "chi2_p_value": float(chi.pvalue),
    }


def numeric_projection(nonces: np.ndarray) -> np.ndarray:
    """Pre-registered exact 32-bit projection for the two-sample KS test."""
    first_four = np.ascontiguousarray(nonces[:, :4])
    values = (
        first_four[:, 0].astype(np.uint64) << 24
        | first_four[:, 1].astype(np.uint64) << 16
        | first_four[:, 2].astype(np.uint64) << 8
        | first_four[:, 3].astype(np.uint64)
    )
    return values.astype(np.float64) / float(1 << 32)


def benjamini_hochberg(p_values: np.ndarray) -> np.ndarray:
    """Benjamini--Hochberg adjusted p-values, implemented without statsmodels."""
    p_values = np.asarray(p_values, dtype=float)
    order = np.argsort(p_values)
    ranked = p_values[order]
    adjusted_ranked = ranked * len(ranked) / np.arange(1, len(ranked) + 1)
    adjusted_ranked = np.minimum.accumulate(adjusted_ranked[::-1])[::-1]
    adjusted = np.empty_like(adjusted_ranked)
    adjusted[order] = np.clip(adjusted_ranked, 0.0, 1.0)
    return adjusted


def session_partition(groups: np.ndarray, seed: int) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Create session-disjoint 70/15/15 partitions for one mode."""
    sessions = np.unique(groups)
    rng = np.random.default_rng(seed)
    rng.shuffle(sessions)
    train_end = int(0.70 * len(sessions))
    validation_end = int(0.85 * len(sessions))
    train_sessions = sessions[:train_end]
    validation_sessions = sessions[train_end:validation_end]
    test_sessions = sessions[validation_end:]
    return (
        np.flatnonzero(np.isin(groups, train_sessions)),
        np.flatnonzero(np.isin(groups, validation_sessions)),
        np.flatnonzero(np.isin(groups, test_sessions)),
    )


def combined_partition(
    normal_groups: np.ndarray,
    anamorphic_groups: np.ndarray,
    seed: int,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    normal_split = session_partition(normal_groups, seed)
    anamorphic_split = session_partition(anamorphic_groups, seed + 10_000)
    offset = len(normal_groups)
    return tuple(
        np.concatenate((normal_split[part], anamorphic_split[part] + offset))
        for part in range(3)
    )


def score_model(
    model: object,
    model_name: str,
    features: np.ndarray,
    labels: np.ndarray,
    train_idx: np.ndarray,
    test_idx: np.ndarray,
    ell: int,
    split_seed: int,
) -> dict[str, float | int | str]:
    model.fit(features[train_idx], labels[train_idx])
    predictions = model.predict(features[test_idx])
    scores = model.predict_proba(features[test_idx])[:, 1]
    auc = float(roc_auc_score(labels[test_idx], scores))
    return {
        "ell": ell,
        "split_seed": split_seed,
        "model": model_name,
        "accuracy": float(accuracy_score(labels[test_idx], predictions)),
        "balanced_accuracy": float(
            balanced_accuracy_score(labels[test_idx], predictions)
        ),
        "roc_auc": auc,
        "auc_advantage": abs(auc - 0.5),
        "train_samples": int(train_idx.size),
        "test_samples": int(test_idx.size),
    }


def classifier_experiments(
    normal: np.ndarray,
    anamorphic: np.ndarray,
    normal_groups: np.ndarray,
    anamorphic_groups: np.ndarray,
    ell: int,
) -> list[dict[str, float | int | str]]:
    features = np.unpackbits(
        np.vstack((normal, anamorphic)), axis=1
    ).astype(np.float32)
    labels = np.concatenate(
        (np.zeros(len(normal), dtype=np.uint8), np.ones(len(anamorphic), dtype=np.uint8))
    )
    rows: list[dict[str, float | int | str]] = []

    for split_seed in SPLIT_SEEDS:
        train_idx, _validation_idx, test_idx = combined_partition(
            normal_groups, anamorphic_groups, split_seed
        )
        logistic = LogisticRegression(
            solver="liblinear",
            max_iter=500,
            random_state=split_seed,
        )
        forest = RandomForestClassifier(
            n_estimators=300,
            max_depth=12,
            min_samples_leaf=10,
            max_features="sqrt",
            class_weight="balanced",
            n_jobs=-1,
            random_state=split_seed,
        )
        rows.append(
            score_model(
                logistic, "Logistic Regression", features, labels,
                train_idx, test_idx, ell, split_seed,
            )
        )
        rows.append(
            score_model(
                forest, "Random Forest", features, labels,
                train_idx, test_idx, ell, split_seed,
            )
        )

    return rows


def summarize_classifier_rows(rows: pd.DataFrame) -> pd.DataFrame:
    summaries: list[dict[str, float | int | str]] = []
    for (ell, model), group in rows.groupby(["ell", "model"], sort=True):
        row: dict[str, float | int | str] = {
            "ell": int(ell),
            "model": str(model),
            "repeated_session_splits": int(len(group)),
        }
        for metric in ("accuracy", "balanced_accuracy", "roc_auc", "auc_advantage"):
            values = group[metric].to_numpy(dtype=float)
            mean = float(values.mean())
            split_sd = float(values.std(ddof=1))
            row[f"{metric}_mean"] = mean
            row[f"{metric}_split_sd"] = split_sd
            row[f"{metric}_split_ci95"] = float(1.96 * split_sd / math.sqrt(len(values)))
        summaries.append(row)
    return pd.DataFrame(summaries)


def plot_detectability(summary: pd.DataFrame, classifier: pd.DataFrame) -> None:
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    data = summary.sort_values("ell")
    fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.2))

    axes[0].plot(
        data["ell"], data["normal_shannon_entropy_bits_per_byte"],
        marker="o", label="Normal",
    )
    axes[0].plot(
        data["ell"], data["anamorphic_shannon_entropy_bits_per_byte"],
        marker="s", label="Anamorphic",
    )
    axes[0].set_ylabel("Shannon entropy (bit/byte)")
    axes[0].set_title("Entropy")
    axes[0].legend(fontsize=8)

    axes[1].plot(data["ell"], data["tv_distance"], marker="o", label="TV distance")
    axes[1].plot(data["ell"], data["ks_statistic"], marker="s", label="KS statistic")
    axes[1].set_ylabel("Distance/statistic")
    axes[1].set_title("Distributional distance")
    axes[1].legend(fontsize=8)

    markers = {"Logistic Regression": "o", "Random Forest": "s"}
    for model, group in classifier.groupby("model"):
        group = group.sort_values("ell")
        axes[2].errorbar(
            group["ell"], group["roc_auc_mean"],
            yerr=group["roc_auc_split_ci95"],
            marker=markers[str(model)], capsize=3, label=f"{model} AUC",
        )
    axes[2].axhline(0.5, color="#7F7F7F", linestyle="--", linewidth=1,
                    label="Random guessing")
    axes[2].set_ylim(0.45, 0.55)
    axes[2].set_ylabel("ROC-AUC")
    axes[2].set_title("Classifier distinguishability")
    axes[2].legend(fontsize=7)

    for axis in axes:
        axis.set_xlabel(r"Hidden bits per record, $\ell$")
        axis.set_xticks(data["ell"])
        axis.grid(True, linestyle="--", alpha=0.3)
    fig.suptitle("Normal versus anamorphic nonce detectability", fontweight="bold")
    fig.tight_layout()
    fig.savefig(
        FIGURE_DIR / "figure_anamorphic_detectability_100k.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.close(fig)


def run() -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(BASE_SEED)
    statistical_rows: list[dict[str, float | int]] = []
    classifier_rows: list[dict[str, float | int | str]] = []
    raw_p_values: list[float] = []
    p_value_locations: list[tuple[int, str]] = []
    start = time.perf_counter()

    for ell in ELL_VALUES:
        ell_start = time.perf_counter()
        dataset_path = OUTPUT_DIR / f"nonce_dataset_ell_{ell}.npz"
        if dataset_path.exists():
            print(f"Loading checkpoint for ell={ell}: {dataset_path}", flush=True)
            with np.load(dataset_path) as dataset:
                normal = dataset["normal_nonces"]
                anamorphic = dataset["anamorphic_nonces"]
                normal_groups = dataset["normal_session_ids"]
                anamorphic_groups = dataset["anamorphic_session_ids"]
                trials = dataset["anamorphic_trials"]
            if len(normal) != N_PER_MODE or len(anamorphic) != N_PER_MODE:
                raise RuntimeError(
                    f"Checkpoint {dataset_path} does not match N_PER_MODE={N_PER_MODE}. "
                    "Delete it before changing the sample size."
                )
        else:
            print(f"Generating N={N_PER_MODE:,} per mode for ell={ell} ...", flush=True)
            normal, normal_groups, _ = generate_dataset("normal", ell, rng)
            anamorphic, anamorphic_groups, trials = generate_dataset("anamorphic", ell, rng)
            np.savez_compressed(
                dataset_path,
                normal_nonces=normal,
                anamorphic_nonces=anamorphic,
                normal_session_ids=normal_groups,
                anamorphic_session_ids=anamorphic_groups,
                anamorphic_trials=trials,
            )

        normal_metrics = distribution_metrics(normal)
        anamorphic_metrics = distribution_metrics(anamorphic)
        normal_probability = byte_counts(normal) / normal.size
        anamorphic_probability = byte_counts(anamorphic) / anamorphic.size
        tv_distance = float(
            0.5 * np.abs(normal_probability - anamorphic_probability).sum()
        )
        normal_bit_positions = np.unpackbits(normal, axis=1).mean(axis=0)
        anamorphic_bit_positions = np.unpackbits(anamorphic, axis=1).mean(axis=0)
        ks = ks_2samp(
            numeric_projection(normal), numeric_projection(anamorphic),
            method="asymp",
        )
        row: dict[str, float | int] = {
            "ell": ell,
            "samples_per_mode": N_PER_MODE,
            **{f"normal_{key}": value for key, value in normal_metrics.items()},
            **{f"anamorphic_{key}": value for key, value in anamorphic_metrics.items()},
            "bit_frequency_absolute_difference": abs(
                normal_metrics["bit_one_frequency"]
                - anamorphic_metrics["bit_one_frequency"]
            ),
            "max_bit_position_frequency_difference": float(
                np.max(np.abs(normal_bit_positions - anamorphic_bit_positions))
            ),
            "ks_statistic": float(ks.statistic),
            "ks_p_value": float(ks.pvalue),
            "tv_distance": tv_distance,
            "anamorphic_mean_trials": float(trials.mean()),
            "anamorphic_median_trials": float(np.median(trials)),
        }
        row_index = len(statistical_rows)
        statistical_rows.append(row)
        for column in ("normal_chi2_p_value", "anamorphic_chi2_p_value", "ks_p_value"):
            raw_p_values.append(float(row[column]))
            p_value_locations.append((row_index, column))

        classifier_rows.extend(
            classifier_experiments(
                normal, anamorphic, normal_groups, anamorphic_groups, ell
            )
        )
        print(
            f"ell={ell} completed in {(time.perf_counter() - ell_start) / 60:.2f} min",
            flush=True,
        )

    adjusted = benjamini_hochberg(np.asarray(raw_p_values))
    for adjusted_value, (row_index, column) in zip(adjusted, p_value_locations):
        statistical_rows[row_index][f"{column}_bh"] = float(adjusted_value)

    statistical = pd.DataFrame(statistical_rows)
    repeated_classifier = pd.DataFrame(classifier_rows)
    classifier_summary = summarize_classifier_rows(repeated_classifier)
    statistical.to_csv(OUTPUT_DIR / "security_metrics_100k.csv", index=False)
    repeated_classifier.to_csv(
        OUTPUT_DIR / "classifier_repeated_splits_100k.csv", index=False
    )
    classifier_summary.to_csv(
        OUTPUT_DIR / "classifier_summary_100k.csv", index=False
    )
    plot_detectability(statistical, classifier_summary)

    metadata = {
        "samples_per_mode_per_ell": N_PER_MODE,
        "ell_values": list(ELL_VALUES),
        "nonce_length_bytes": NONCE_LEN,
        "records_per_session": RECORDS_PER_SESSION,
        "split_seeds": list(SPLIT_SEEDS),
        "python": sys.version,
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scipy": scipy.__version__,
        "scikit_learn": sklearn.__version__,
        "elapsed_minutes": (time.perf_counter() - start) / 60,
    }
    (OUTPUT_DIR / "metadata_100k.json").write_text(
        json.dumps(metadata, indent=2), encoding="utf-8"
    )
    print("\nStatistical summary:")
    print(
        statistical[[
            "ell", "tv_distance", "ks_statistic", "ks_p_value",
            "ks_p_value_bh", "bit_frequency_absolute_difference",
        ]].to_string(index=False)
    )
    print("\nClassifier summary:")
    print(
        classifier_summary[[
            "ell", "model", "accuracy_mean", "roc_auc_mean",
            "roc_auc_split_ci95", "auc_advantage_mean",
        ]].to_string(index=False)
    )
    print(f"\nResults: {OUTPUT_DIR.resolve()}")
    print(f"Figure:  {FIGURE_DIR.resolve()}")


if __name__ == "__main__":
    run()

Generating N=100,000 per mode for ell=1 ...
ell=1 completed in 11.25 min
Generating N=100,000 per mode for ell=2 ...
ell=2 completed in 11.32 min
Generating N=100,000 per mode for ell=4 ...
ell=4 completed in 11.38 min
Generating N=100,000 per mode for ell=8 ...
ell=8 completed in 14.17 min

Statistical summary:
 ell  tv_distance  ks_statistic  ks_p_value  ks_p_value_bh  bit_frequency_absolute_difference
   1     0.007818       0.00500    0.163534       0.857115                           0.000097
   2     0.007690       0.00359    0.538471       0.857115                           0.000259
   4     0.008083       0.00420    0.340053       0.857115                           0.000026
   8     0.007673       0.00365    0.516893       0.857115                           0.000213

Classifier summary:
 ell               model  accuracy_mean  roc_auc_mean  roc_auc_split_ci95  auc_advantage_mean
   1 Logistic Regression       0.498903      0.499489            0.001856            0.002269
   1   